---
title : "Talk with WISKI"
execute:
    error: false
    eval: false
    output: asis
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

# Quality Code

In [1]:
# zorg dat alle variables goed staan
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

In [2]:
from talk_with_wiski.utils.logging_utils import setup_logging, logger
from pathlib import Path

logging_path = setup_logging(log_level="WARN", log_dir=str(Path.cwd()))
# bijvoobeeld @Andries:
logger.info(f"Logging path: {logging_path}")

In [3]:
# to do analysis on timeseries
from talk_with_wiski.models import (
    AnalyseTimeseriesQualityCodeInput,
)
from talk_with_wiski.tools import (
    get_timeseries_list,
    analyse_timeseries_quality_code,
)

import pandas as pd

In [4]:
# for statistics and trend, same object can be used
analyse_timeseries_input = AnalyseTimeseriesQualityCodeInput(
    ts_id=["37481042", "63160042"],  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P1D",  # dit mag niet icm from_date.. to_date
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
)
out_qualcode = await analyse_timeseries_quality_code(ctx, analyse_timeseries_input)
df = pd.DataFrame(out_qualcode.timeseries_quality_codes)
print(df)

      ts_id  station_name    station_no parametertype_name  n_datapoints  \
0  37481042  Bovenstrooms  1198_WTH_BOV            WNS8487            25   
1  63160042         Pomp1  BRE0301_PMP1            WNS9030            22   

   Ongecontroleerd  AutoValide (oud)  
0               25                 0  
1                0                22  


# Metadata voor TimeseriesType

In [5]:
ts_type_id_in = [6042, 14042]  # zijn de type_ids voor WNS*

In [6]:
# zorg dat alle variable goed staan
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

In [7]:
from talk_with_wiski.models import GetTimeseriesTypeInput, GetTimeseriesTypeOutput
from talk_with_wiski.tools import get_timeseries_type

for idx in [0, 1]:
    get_timeseries_type_input = GetTimeseriesTypeInput(ts_type_id=ts_type_id_in[idx])
    get_timeseries_type_output: GetTimeseriesTypeOutput = await get_timeseries_type(
        ctx, get_timeseries_type_input
    )
    data = get_timeseries_type_output.model_dump()

    unique_ts_names = {x["ts_name"] for x in data["timeseries_type_list"]}
    print(unique_ts_names)

{'Handmeting.B4', 'Handmeting.B2', 'ContinueMeting.F', 'ContinueMeting.P', 'Handmeting'}
{'Maand.Gem', 'Day.Mean', 'Jaar.Gem', 'Uur.Gem', 'Dag.Gem'}


# Metadata voor {stations, sites, ts_id}
# obv request naar get*List, met returnfields die metadata bevatten
# mss moet ts_type_id hier ook nog bij? 

In [8]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput

get_timeseries_list_input = GetTimeseriesListInput(
    # parametertype_name='WNS*',  # dit geeft een leeg resultaat
    # parametertype_name=['WNS9688','WNS1919','WNS8487'], # en ook voor een lijst lijkt dit niet te werken
    parametertype_name=["WNS8487"],
    returnfields="station_name,station_no,station_id,ts_id,ts_name,parametertype_id,parametertype_name,ts_type_id,ts_type_name",
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

data = get_timeseries_list_output.model_dump()

ts_id_list = list({x["ts_id"] for x in data["data"]})
print(ts_id_list)
ts_id_list = ts_id_list[:10]  # reduce to first 10

ts_ids_string = ",".join(ts_id_list)
print(ts_ids_string)

['58591042', '37372042', '15230042', '37968042', '80775042', '77222042', '3128042', '37706042', '84547042', '60059042', '5012042', '137042', '60033042', '9581042', '4962042', '50453042', '9285042', '9078042', '27419042', '48388042', '78948042', '15222042', '72859042', '5274042', '79617042', '4268042', '70066042', '9115042', '86382042', '78845042', '59965042', '8451042', '7959042', '2262042', '37665042', '80992042', '768042', '80916042', '3858042', '1263042', '26080042', '83711042', '74161042', '4546042', '88312042', '79708042', '17130042', '9582042', '476042', '85639042', '9379042', '9388042', '31824042', '9123042', '8430042', '21532042', '59915042', '59404042', '2442042', '62241042', '85751042', '93166042', '59777042', '1877042', '25714042', '17114042', '4496042', '4089042', '1830042', '31420042', '9204042', '58550042', '4730042', '71100042', '4397042', '16660042', '85724042', '80708042', '1824042', '81484042', '72976042', '8806042', '58882042', '32901042', '5703042', '5415042', '9477

In [9]:
from talk_with_wiski.models import GetMetadataInput, GetMetadataOutput
from talk_with_wiski.tools import get_metadata

input = GetMetadataInput(
    ts_id=ts_id_list,
)
ts_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)

print(ts_metadata_list_of_dicts.data[0])

# Extract site_no values from the metadata
site_no_list = [item["site_no"] for item in ts_metadata_list_of_dicts.data]
print(f"sites: {site_no_list}")

# Extract site_no values from the metadata
station_no_list = [item["station_no"] for item in ts_metadata_list_of_dicts.data]
print(f"stations: {station_no_list}")

input = GetMetadataInput(
    site_no=site_no_list,
)
site_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)
print(site_metadata_list_of_dicts.data)

input = GetMetadataInput(
    station_no=station_no_list,
)
station_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)
print(station_metadata_list_of_dicts.data)

# Voorbeeld voor koppelen van twee tabellen metadata (site_no based)
merged_list = [
    {**d1, **d2}
    for d1 in ts_metadata_list_of_dicts.data
    for d2 in site_metadata_list_of_dicts.data
    if d1["site_no"] == d2["site_no"]
]
print(merged_list[0])

print(len(ts_id_list), len(site_no_list), len(merged_list))

{'ts_id': '80775042', 'site_no': '2279', 'station_no': '2279_WTH', 'catchment_id': '0', 'Meetdoel_BOS_hoog_water': 'no', 'STATION_NUMMER_OUD': '2279_WTH', 'station_diary': '', 'Meetdoel_Mark-Vliet_boezem': 'no', 'Meetdoel_basismeetnet_freatisch_G': 'no', 'Meetdoel_project_onderzoek': 'yes', 'Meetdoel_KRW': 'no', 'Meetdoel_peilbeheer': 'no', 'STATUS': 'Observed', 'ngr_letter': '-', 'Meetdoel_calamiteitenzorg': 'no', 'admin_level': 'District', 'region_district': '---', 'Meetdoel_Waterveiligheid_projecten': 'no', 'Meetdoel_Waterveiligheid_primair': 'no', 'Meetdoel_controlemetingen': 'no', 'Meetdoel_BMV': 'yes', 'Meetfrequentie_Handmeting': '', 'Meetdoel_ondergrens_N2000': 'no', 'Meetdoel_Bedienen': 'no', 'Meetdoel_basismeetnet_N': 'no', 'MEETWIJZEBRON': 'Telemetrie MOS', 'station_diary_status': '', 'Meetdoel_extern': 'no', 'Meetdoel_waterakkoorden': 'no', 'Meetdoel_Waterkwaliteit': 'no', 'Meetdoel_Waterveiligheid_regionaal': 'no', 'Meetdoel_basismeetnet_Q': 'no', 'Meetdoel_Waterketen': 'n